# reglscatterpy

Interactive WebGL scatterplots for single-cell and spatial data. Each cell is a short, self-contained example. Run it top to bottom.

In [ ]:
import scanpy as sc
import reglscatterpy as rs

adata = sc.datasets.pbmc3k_processed()   # downloads once, 2638 PBMCs
adata

## Basic plots

Color by a cell-type column, or by a gene name.

In [ ]:
rs.scatterplot(adata, basis="umap", color_by="louvain")   # categorical column

In [ ]:
rs.scatterplot(adata, basis="umap", color_by="CST3")      # a gene

A pandas DataFrame works too. Give the coordinate columns with x and y.

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "x": adata.obsm["X_umap"][:, 0],
    "y": adata.obsm["X_umap"][:, 1],
    "cluster": adata.obs["louvain"].values,
})
rs.scatterplot(df, x="x", y="y", color_by="cluster")

## Select cells and read them back

Use interactive=True to get the live widget, then lasso cells and use them in Python. Here the selection is set from code so the notebook runs without a mouse.

In [ ]:
w = rs.scatterplot(adata, basis="umap", color_by="louvain", interactive=True)
w.selection = adata.obs_names[adata.obs["louvain"] == "B cells"]
print(len(w.selection), "cells selected")
w

Subset the data to the selection.

In [ ]:
adata[w.selection]        # same as w.subset()

Count the categories in the selection.

In [ ]:
w.composition("louvain")

Write a label onto the selected cells.

In [ ]:
w.annotate("cell_type", "B cells")
adata.obs["cell_type"].value_counts(dropna=False)

## Differential expression

Get the top marker genes of the selection against the rest. The result is scanpy's native rank_genes_groups, saved to adata.uns.

In [ ]:
w.diff_expression(n=10)
sc.get.rank_genes_groups_df(adata, group="A").head(10)

Pick the backend with engine. The default uses scanpy. Use gpu for a cupy Welch t-test, or ttest for the built-in CPU test.

In [ ]:
w.diff_expression(n=10, engine="gpu")     # needs cupy; engine="scanpy" to require scanpy

Split the selection by an obs column and compare its levels.

In [ ]:
import numpy as np
adata.obs["group"] = pd.Categorical(np.where(adata.obs["n_genes"] > 800, "high", "low"))
w.selection = list(range(adata.n_obs))
res = w.diff_expression_by("group")
res["names"].dtype.names

## Multiple panels

Pass a list of genes or columns to get a linked grid. Pan, zoom and lasso stay in sync across panels.

In [ ]:
rs.scatterplot(adata, basis="umap", color_by=["louvain", "CST3", "NKG7"], ncols=3)

A list of embeddings works the same way.

In [ ]:
rs.scatterplot(adata, basis=["umap", "pca"], color_by="louvain")

Or build the plots first and combine them with compose.

In [ ]:
from reglscatterpy import scatterplot, compose
a = scatterplot(adata, basis="umap", color_by="louvain")
b = scatterplot(adata, basis="pca", color_by="louvain")
compose([a, b])

## Spatial data

Anything with obsm['spatial'] plots with basis='spatial'. This uses a public Visium dataset from squidpy. Install it first with pip install squidpy.

In [ ]:
import squidpy as sq
sp = sq.datasets.visium_hne_adata()   # mouse brain; has spatial and X_umap

In [ ]:
rs.scatterplot(sp, basis="spatial", color_by="cluster")   # tissue map

In [ ]:
rs.scatterplot(sp, basis="spatial", color_by="Olfm1")     # a gene in the tissue

Morph animates the same cells from the UMAP into their tissue positions.

In [ ]:
w_sp = rs.scatterplot(sp, basis="umap", color_by="cluster", interactive=True)
w_sp

In [ ]:
w_sp.morph_to("spatial")   # run in its own cell; w_sp.morph_to("umap") goes back

## Save a standalone HTML

Write one self-contained file that opens in any browser with no kernel.

In [ ]:
rs.save_html(w, "umap.html")   # or w.to_html("umap.html")